# Observable Gap Assessment

This notebook investigates *why* existing instrumentation techniques miss
the traffic they miss. We evaluate two candidate explanations:
certificate pinning and custom TLS library bundling. The analysis
demonstrates that unobserved traffic can arise from the interplay of multiple
factors rather than a single cause, and behavioural profiles constructed
solely from certificate pinning detection may give an incomplete picture.

Further, the results drawn from such analysis offer an incomplete picture, 
and can hence be considered unreliable.

**Paper reference:** $\S$ 6.3, Figure 3

In [1]:

import pandas as pd
from analysis_utils.shared_setup import map_libs, mark_print
from tabulate import tabulate

http_df = pd.read_parquet("../data/http_requests.parquet")
pii_df = pd.read_parquet("../data/pii_observed.parquet")
pinning_df = pd.read_parquet("../data/pinning.parquet")
print(f"pinning_df.parquet DF cols: {pinning_df.columns} | {pinning_df.size}")

advaita_df = pd.read_parquet("../data/observable_gaps.parquet")
print(f"observable_gaps.parquet DF cols: {advaita_df.columns} | {advaita_df.size}")

advaita_df = map_libs(advaita_df)
advaita_hosts = set(advaita_df["host"].unique())

pinned_hosts = set(pinning_df["host"].unique())

advaita_df["valid"] = advaita_df.apply(lambda r: r["appname"] in r["app_test"], axis=1)
advaita_df = advaita_df[advaita_df["valid"]].copy()
advaita_df = advaita_df[advaita_df["host"].notna()]

host_to_company = http_df.copy().set_index("host")["company"].to_dict()
advaita_df["company"] = advaita_df["company"].fillna(advaita_df["host"].map(host_to_company))

pii_clean = pii_df.drop(columns=[
    "app", "port", "dir", "pii_label", "vantage_type", "ip",
    "time", "decoders", "http_stream_type", "app_test", "tls_lib",
])
pii_clean = pii_clean[pii_clean["clientrandom"].notna()].drop_duplicates()

pii_by_random = pii_clean.groupby("clientrandom")["pii_type"].apply(
    lambda x: ", ".join(sorted(x.unique()))
).to_dict()
advaita_df["pii_types"] = advaita_df["clientrandom"].map(pii_by_random)

mark_print(
    f"Loaded **{len(advaita_df):,}** Advaita observations across "
    f"**{len(advaita_hosts):,}** unique hosts"
)

pinning_df.parquet DF cols: Index(['host', 'vantage_type', 'ip', 'app_test', 'mitm_lines', 'uid', 'pid',
       'app', 'comm', 'logcat_evidence', 'company', 'pinning_type_label'],
      dtype='str') | 7188
observable_gaps.parquet DF cols: Index(['uuid', 'clientrandom', 'app_test', 'connection_id', 'stream_id',
       'direction', 'source', 'timestamp_us', 'host', 'host_src', 'ip', 'port',
       'frame_number', 'appname', 'uid', 'pid', 'comm', 'is_quic', 'protocol',
       'http_stream_type', 'tls_lib', 'total_bytes', 'company'],
      dtype='str') | 506253


Loaded **21,883** Advaita observations across **1,628** unique hosts

### Coverage Gaps Across Existing Techniques

Identify hosts observed by Advaita but missed by each existing
instrumentation technique, and the intersection missed by all three.

In [2]:
# Missed host computation
techniques = ["Aosp-Sci", "Frida", "Mitm"]
advaita_df = advaita_df[advaita_df["company"].notna()]

missing = {
    vp: advaita_hosts - set(http_df[http_df["vantage_type"] == vp]["host"].unique())
    for vp in techniques
}
all_missing = missing["Aosp-Sci"] & missing["Frida"] & missing["Mitm"]

missed_rows = []
for vp in techniques:
    missed_rows.append({
        "Technique": vp,
        "Hosts Missed": len(missing[vp]),
        "% of Advaita Hosts": f"{len(missing[vp]) / len(advaita_hosts) * 100:.1f}%",
    })
missed_rows.append({
    "Technique": "All three",
    "Hosts Missed": len(all_missing),
    "% of Advaita Hosts": f"{len(all_missing) / len(advaita_hosts) * 100:.1f}%",
})

mark_print("**Hosts missed per existing technique:**")
mark_print(tabulate(missed_rows, headers="keys", tablefmt="pipe", showindex=False))

**Hosts missed per existing technique:**

| Technique   |   Hosts Missed | % of Advaita Hosts   |
|:------------|---------------:|:---------------------|
| Aosp-Sci    |           1058 | 65.0%                |
| Frida       |           1005 | 61.7%                |
| Mitm        |            891 | 54.7%                |
| All three   |            729 | 44.8%                |

### Traffic Volume of Missed Hosts

Quantify the traffic volume represented by missed hosts to establish
that the gap is not marginal.

In [3]:
# Cell: Traffic gap
advaita_drs = http_df[http_df["host"].isin(advaita_hosts)]["company"].nunique()
missed_bytes = advaita_df[advaita_df["host"].isin(all_missing)]["total_bytes"].sum()
total_bytes = advaita_df["total_bytes"].sum()
missed_pct = missed_bytes / total_bytes * 100
missed_with_pii = advaita_df[
    (advaita_df["host"].isin(all_missing)) & (advaita_df["pii_types"].notna())
]["host"].nunique()

mark_print(
    f"Of the **{len(advaita_hosts):,}** unique hosts contacted by "
    f"**{advaita_drs:,}** DRs, **{len(all_missing):,}** "
    f"({len(all_missing) / len(advaita_hosts) * 100:.1f}%) were missed by all "
    f"three existing techniques. These hosts account for "
    f"**{missed_bytes / 1e6:.2f} MB** ({missed_pct:.2f}%) of total observed "
    f"traffic. **{missed_with_pii}** of the missed hosts transmit PII."
)

Of the **1,628** unique hosts contacted by **378** DRs, **729** (44.8%) were missed by all three existing techniques. These hosts account for **1133.67 MB** (23.07%) of total observed traffic. **167** of the missed hosts transmit PII.

### DR-Level Impact: Fully Captured, Partially Captured, Fully Missed

Classify each data recipient by whether all, some, or none of its
hosts were captured by the existing instrumentation techniques.

In [4]:
fully_captured = []
partially_captured = []
fully_missed = []

for company in advaita_df["company"].dropna().unique():
    company_hosts = set(advaita_df[advaita_df["company"] == company]["host"].unique())
    missed = company_hosts & all_missing
    if len(missed) == 0:
        fully_captured.append(company)
    elif len(missed) == len(company_hosts):
        fully_missed.append(company)
    else:
        partially_captured.append(company)

dr_class_rows = [
    {"Category": "Fully captured", "DRs": len(fully_captured)},
    {"Category": "Partially captured (distorted view)", "DRs": len(partially_captured)},
    {"Category": "Fully missed", "DRs": len(fully_missed)},
]

mark_print("**DR-level capture classification:**")
mark_print(tabulate(dr_class_rows, headers="keys", tablefmt="pipe", showindex=False))

**DR-level capture classification:**

| Category                            |   DRs |
|:------------------------------------|------:|
| Fully captured                      |   165 |
| Partially captured (distorted view) |    97 |
| Fully missed                        |   118 |

In [5]:
# DR-level TLS classification
system_label = "Android System"
webview_label = "WebView"

missed_dr_libs = advaita_df[advaita_df["host"].isin(all_missing)].groupby("company")["tls_lib_label"].apply(
    lambda x: set(str(v) for v in x if pd.notna(v))
)

dr_custom_exclusive = sum(
    1 for labels in missed_dr_libs if labels
    and not any(system_label in l or webview_label in l for l in labels)
)
dr_webview_exclusive = sum(
    1 for labels in missed_dr_libs if labels
    and any(webview_label in l for l in labels)
    and not any(l for l in labels if system_label not in l and webview_label not in l)
)
dr_system_exclusive = sum(
    1 for labels in missed_dr_libs if labels
    and all(system_label in l for l in labels)
)
dr_mixed = len(missed_dr_libs) - dr_custom_exclusive - dr_webview_exclusive - dr_system_exclusive
total_missed_drs = len(missed_dr_libs)

dr_class_rows = [
    {"TLS Category": "Custom TLS (exclusive)", "DRs": dr_custom_exclusive,
     "%": f"{dr_custom_exclusive / total_missed_drs * 100:.1f}%"},
    {"TLS Category": "WebView BoringSSL (exclusive)", "DRs": dr_webview_exclusive,
     "%": f"{dr_webview_exclusive / total_missed_drs * 100:.1f}%"},
    {"TLS Category": "System Conscrypt (exclusive)", "DRs": dr_system_exclusive,
     "%": f"{dr_system_exclusive / total_missed_drs * 100:.1f}%"},
    {"TLS Category": "Mixed (multiple libraries)", "DRs": dr_mixed,
     "%": f"{dr_mixed / total_missed_drs * 100:.1f}%"},
    {"TLS Category": "Total", "DRs": total_missed_drs, "%": ""},
]

mark_print("**DR-level TLS classification of missed DRs:**")
mark_print(tabulate(dr_class_rows, headers="keys", tablefmt="pipe", showindex=False))

**DR-level TLS classification of missed DRs:**

| TLS Category                  |   DRs | %     |
|:------------------------------|------:|:------|
| Custom TLS (exclusive)        |    17 | 7.9%  |
| WebView BoringSSL (exclusive) |   150 | 69.8% |
| System Conscrypt (exclusive)  |    23 | 10.7% |
| Mixed (multiple libraries)    |    25 | 11.6% |
| Total                         |   215 |       |

### Conscrypt vs. Non-System Miss Rates

Compare miss rates across library types. If the gap were caused by
certificate pinning or network-level issues, we would expect similar
miss rates across library types. A disproportionate miss rate for
non-system libraries confirms bundled TLS as the mechanism.

In [6]:
# --- Cell: Miss rate comparison ---
all_conscrypt_hosts = set(
    advaita_df[advaita_df["tls_lib_label"].str.contains("Conscrypt", case=False, na=False)]["host"].unique()
)
all_webview_hosts = set(
    advaita_df[advaita_df["tls_lib_label"].str.contains(webview_label, case=False, na=False)]["host"].unique()
)
all_nonsystem_hosts = set(
    advaita_df[~advaita_df["tls_lib_label"].str.contains(system_label, case=False, na=False)]["host"].unique()
)

any_technique_missed = missing["Mitm"] | missing["Frida"] | missing["Aosp-Sci"]
conscrypt_missed_by_any = all_conscrypt_hosts & any_technique_missed
webview_missed_by_any = all_webview_hosts & any_technique_missed
nonsystem_missed_by_any = all_nonsystem_hosts & any_technique_missed

rate_rows = [
    {"Library Type": "System Conscrypt", "Total Hosts": len(all_conscrypt_hosts),
     "Missed (≥1 technique)": len(conscrypt_missed_by_any),
     "Miss Rate": f"{len(conscrypt_missed_by_any) / len(all_conscrypt_hosts) * 100:.1f}%"},
    {"Library Type": "WebView BoringSSL", "Total Hosts": len(all_webview_hosts),
     "Missed (≥1 technique)": len(webview_missed_by_any),
     "Miss Rate": f"{len(webview_missed_by_any) / len(all_webview_hosts) * 100:.1f}%"},
    {"Library Type": "All non-system", "Total Hosts": len(all_nonsystem_hosts),
     "Missed (≥1 technique)": len(nonsystem_missed_by_any),
     "Miss Rate": f"{len(nonsystem_missed_by_any) / len(all_nonsystem_hosts) * 100:.1f}%"},
]

mark_print("**Miss rates by TLS library type:**")
mark_print(tabulate(rate_rows, headers="keys", tablefmt="pipe", showindex=False))

**Miss rates by TLS library type:**

| Library Type      |   Total Hosts |   Missed (≥1 technique) | Miss Rate   |
|:------------------|--------------:|------------------------:|:------------|
| System Conscrypt  |           926 |                     545 | 58.9%       |
| WebView BoringSSL |           785 |                     676 | 86.1%       |
| All non-system    |          1117 |                     915 | 81.9%       |

### Certificate Pinning as Alternative Explanation

If certificate pinning explained the observability gap, we would
expect missed hosts to disproportionately implement pinning. We test
this by checking how many missed hosts appear in the pinning dataset.
Additionally, Frida with objection — designed specifically to bypass
pinning — misses the same hosts as mitmproxy, further ruling out
pinning as the causal mechanism.

In [7]:
# --- Cell: Pinning analysis ---
missed_and_pinned = len(all_missing & pinned_hosts)
total_missed = len(all_missing)
mark_print(
    f"Of the **{total_missed:,}** hosts missed by all three existing techniques, "
    f"only **{missed_and_pinned}** ({missed_and_pinned / total_missed * 100:.1f}%) "
    f"implement certificate pinning. Frida with objection, designed specifically "
    f"to bypass pinning, misses the same hosts as mitmproxy. Certificate pinning "
    f"does not explain the observability gap."
)

Of the **729** hosts missed by all three existing techniques, only **37** (5.1%) implement certificate pinning. Frida with objection, designed specifically to bypass pinning, misses the same hosts as mitmproxy. Certificate pinning does not explain the observability gap.

### ByteDance Traffic Gap

ByteDance apps (TikTok, CapCut) bundle custom BoringSSL forks.
We compare what Advaita observes against what mitmproxy captures
to quantify the traffic invisible to proxy-based interception.

In [8]:
# --- Cell: ByteDance case study ---
# All ByteDance hosts observed by Advaita
bytedance_hosts = set(
    advaita_df[advaita_df["company"].str.contains("bytedance", case=False, na=False)]["host"].unique()
)

# ByteDance hosts captured by mitmproxy
mitm_bytedance = set(
    http_df[
        (http_df["vantage_type"] == "Mitm")
        & http_df["company"].str.contains("bytedance", case=False, na=False)
    ]["host"].unique()
)

captured = bytedance_hosts & mitm_bytedance
missed = bytedance_hosts - mitm_bytedance

mark_print(
    f"**ByteDance host coverage:** Advaita observes **{len(bytedance_hosts)}** hosts. "
    f"Mitmproxy captures **{len(captured)}**, misses **{len(missed)}** "
    f"({len(missed) / len(bytedance_hosts) * 100:.1f}%)."
)

# Traffic volume and TLS library breakdown of missed hosts
missed_tx = advaita_df[
    advaita_df["host"].isin(missed)
    & advaita_df["company"].str.contains("bytedance", case=False, na=False)
].drop_duplicates(subset=["clientrandom"]).copy()

total_missed_bytes = missed_tx["total_bytes"].sum()

lib_breakdown = (
    missed_tx.groupby("tls_lib_label")
    .agg(
        sessions=("clientrandom", "count"),
        bytes_mb=("total_bytes", lambda x: x.sum() / 1e6),
    )
    .sort_values("bytes_mb", ascending=False)
    .reset_index()
    .rename(columns={"tls_lib_label": "TLS Library", "bytes_mb": "Traffic (MB)"})
)

custom_labels = missed_tx[
    missed_tx["tls_lib_label"].str.contains("TikTok|CapCult", case=False, na=False)
]
custom_bytes = custom_labels["total_bytes"].sum()

mark_print(
    f"**Missed ByteDance traffic:** {total_missed_bytes / 1e6:.2f} MB across "
    f"**{len(missed)}** hosts. Of this, **{custom_bytes / 1e6:.2f} MB** "
    f"({custom_bytes / total_missed_bytes * 100:.1f}%) flows through "
    f"TikTok/CapCut's custom BoringSSL forks."
)

mark_print("**TLS library breakdown of missed ByteDance traffic:**")
mark_print(tabulate(lib_breakdown, headers="keys", tablefmt="pipe", showindex=False))

**ByteDance host coverage:** Advaita observes **109** hosts. Mitmproxy captures **41**, misses **68** (62.4%).

**Missed ByteDance traffic:** 202.83 MB across **68** hosts. Of this, **192.01 MB** (94.7%) flows through TikTok/CapCut's custom BoringSSL forks.

**TLS library breakdown of missed ByteDance traffic:**

| TLS Library                        |   sessions |   Traffic (MB) |
|:-----------------------------------|-----------:|---------------:|
| TikTok (BoringSSL)                 |        244 |      186.323   |
| Android WebView (Static BoringSSL) |         22 |        6.46438 |
| CapCult (BoringSSL)                |         14 |        5.68758 |
| GMS (Conscrypt)                    |         15 |        2.25023 |
| Android System (Conscrypt)         |         17 |        1.89385 |

### Custom TLS Libraries

Per-company, per-library breakdown of host counts, traffic volume,
capture rates by each existing technique, and PII exposure in missed
traffic.

In [ ]:
from analysis_utils.shared_setup import GAPS_DR, mark_print

company_lib_stats = []

for company in advaita_df["company"].dropna().unique():
    company_df = advaita_df[advaita_df["company"] == company]

    for lib in company_df["tls_lib_label"].dropna().unique():
        lib_df = company_df[company_df["tls_lib_label"] == lib]
        lib_hosts = set(lib_df["host"].unique())
        lib_bytes = lib_df["total_bytes"].sum()
        missed = lib_hosts & all_missing

        all_piis = lib_df["pii_types"].dropna().unique()
        all_piis_flat = ", ".join(sorted(set(
            p for piis in all_piis for p in piis.split(", ")
        ))) if len(all_piis) > 0 else ""

        missed_df = lib_df[lib_df["host"].isin(missed)]
        missed_piis = missed_df["pii_types"].dropna().unique()
        missed_piis_flat = ", ".join(sorted(set(
            p for piis in missed_piis for p in piis.split(", ")
        ))) if len(missed_piis) > 0 else ""

        missed_pinned = missed & pinned_hosts

        company_lib_stats.append({
            "company": company,
            "tls_lib": lib,
            "hosts": len(lib_hosts),
            "bytes_mb": lib_bytes / 1e6,
            "captured_mitm": len(lib_hosts - missing["Mitm"]),
            "captured_frida": len(lib_hosts - missing["Frida"]),
            "captured_aospsci": len(lib_hosts - missing["Aosp-Sci"]),
            "missed_all": len(missed),
            "missed_pii": missed_piis_flat,
            "missed_and_pinned": len(missed_pinned),
            "pii_types": all_piis_flat,
        })

clib_table = pd.DataFrame(company_lib_stats).sort_values(
    ["company", "missed_all"], ascending=[True, False]
)

multi_lib = clib_table.groupby("company")["tls_lib"].nunique()
multi_lib = multi_lib[multi_lib > 1]
companies_with_missed_pii = clib_table[clib_table["missed_pii"] != ""]["company"].nunique()

mark_print(
    f"**{len(multi_lib)}** DRs route traffic through multiple TLS libraries. "
    f"**{companies_with_missed_pii}** DRs transmit PII exclusively through "
    f"non-system libraries invisible to all existing techniques."
)

top_companies = clib_table.groupby('company').agg(
    total_missed=('missed_all', 'sum'),
    total_bytes=('bytes_mb', 'sum')
).sort_values('total_missed', ascending=False).head(6).index

subset = clib_table[clib_table['company'].isin(GAPS_DR)].copy()
subset = subset.sort_values(['company', 'missed_all'], ascending=[True, False])
mark_print(tabulate(subset[subset['company'].isin(top_companies)], headers="keys", tablefmt="pipe", showindex=False))
subset[subset['company'].isin(top_companies)]


**139** DRs route traffic through multiple TLS libraries. **55** DRs transmit PII exclusively through non-system libraries invisible to all existing techniques.

| company      | tls_lib                            |   hosts |   bytes_mb |   captured_mitm |   captured_frida |   captured_aospsci |   missed_all | missed_pii                                                                                              |   missed_and_pinned | pii_types                                                                                                              |
|:-------------|:-----------------------------------|--------:|-----------:|----------------:|-----------------:|-------------------:|-------------:|:--------------------------------------------------------------------------------------------------------|--------------------:|:-----------------------------------------------------------------------------------------------------------------------|
| ByteDance    | TikTok (BoringSSL)                 |      43 | 223.142    |               3 |                3 |                  4 |           39 | Advertising ID, Android ID, Contact Email, Contact Phone, Firebase Installation ID, Google Account Name |                   8 | Advertising ID, Android ID, Contact Email, Contact Phone, Firebase Installation ID, Google Account Name                |
| ByteDance    | CapCult (BoringSSL)                |      28 | 227.63     |              17 |               12 |                  3 |           10 | Android ID, Gmail Profile Picture, Google Account Email, Google Account Name                            |                   0 | Advertising ID, Android ID, Firebase Installation ID, Gmail Profile Picture, Google Account Email, Google Account Name |
| ByteDance    | Android WebView (Static BoringSSL) |      17 |  45.8266   |               8 |                6 |                  8 |            5 |                                                                                                         |                   0 |                                                                                                                        |
| ByteDance    | GMS (Conscrypt)                    |       7 |  21.086    |               4 |                5 |                  5 |            2 |                                                                                                         |                   0 | Advertising ID, App Set ID                                                                                             |
| ByteDance    | Android System (Conscrypt)         |      12 |  39.301    |               7 |                7 |                  9 |            2 | App Set ID                                                                                              |                   0 | Advertising ID, App Set ID                                                                                             |
| Google       | Android WebView (Static BoringSSL) |      39 |  72.2577   |              23 |               15 |                 14 |           13 | Advertising ID, Android ID, App Set ID, Google Account Email, Location                                  |                   1 | Advertising ID, Android ID, App Set ID, Contact Phone, Google Account Email, Location                                  |
| Google       | Android System (Conscrypt)         |      25 |  19.5525   |              18 |               11 |                 14 |            5 |                                                                                                         |                   0 | Gmail Profile Picture, Google Account Email, Google Account Name                                                       |
| Google       | CapCult (BoringSSL)                |       6 |   0        |               1 |                1 |                  0 |            5 | Android ID, Gmail Profile Picture, Google Account Email, Google Account Name                            |                   0 | Android ID, Gmail Profile Picture, Google Account Email, Google Account Name                                           |
| Google       | Adwords (Flutter)                  |       5 |   2.40256  |               3 |                3 |                  2 |            2 | Advertising ID                                                                                          |                   0 | Advertising ID                                                                                                         |
| Google       | Snapchat (Static BoringSSL)        |       3 |   0.916838 |               1 |                1 |                  1 |            2 | Advertising ID, Android ID                                                                              |                   0 | Advertising ID, Android ID                                                                                             |
| Google       | YouTube (Cronet)                   |       7 |   1.1393   |               6 |                3 |                  2 |            1 |                                                                                                         |                   0 |                                                                                                                        |
| Google       | GMS (Conscrypt)                    |      11 |   4.84635  |              11 |                8 |                  6 |            0 |                                                                                                         |                   0 | Firebase Installation ID, GSF ID, Gmail Profile Picture, Google Account Email, Google Account Name                     |
| Google AdMob | Android WebView (Static BoringSSL) |      41 |  22.3509   |               4 |                2 |                  3 |           37 |                                                                                                         |                   0 | Advertising ID                                                                                                         |
| Google AdMob | Android System (Conscrypt)         |      39 |   1.8624   |               3 |                2 |                  3 |           36 |                                                                                                         |                   0 |                                                                                                                        |
| Google AdMob | GMS (Conscrypt)                    |       2 |   0.055957 |               2 |                2 |                  2 |            0 |                                                                                                         |                   0 |                                                                                                                        |
| Google AdMob | Adwords (Flutter)                  |       1 |   0.276669 |               1 |                1 |                  1 |            0 |                                                                                                         |                   0 |                                                                                                                        |
| Meta         | Android System (Conscrypt)         |      16 | 247.031    |               6 |                6 |                  5 |           10 |                                                                                                         |                   0 | Advertising ID, Google Account Email                                                                                   |
| Meta         | Meta (Instagram - Fizz)            |       9 | 215.565    |               0 |                0 |                  0 |            9 | Advertising ID, Android ID, Photos Location, Router Location, Video Location                            |                   3 | Advertising ID, Android ID, Photos Location, Router Location, Video Location                                           |
| Meta         | Android WebView (Static BoringSSL) |      12 |  12.8857   |               8 |                4 |                  3 |            4 |                                                                                                         |                   0 | Google Account Email                                                                                                   |
| Meta         | Meta (Facebook - Fizz)             |       4 |   1.69849  |               1 |                1 |                  1 |            3 |                                                                                                         |                   0 | Router Location                                                                                                        |
| Meta         | GMS (Conscrypt)                    |       6 | 126.773    |               6 |                6 |                  5 |            0 |                                                                                                         |                   0 | Advertising ID                                                                                                         |
| YouTube      | Android WebView (Static BoringSSL) |      34 | 124.626    |              10 |                3 |                  3 |           24 |                                                                                                         |                   1 |                                                                                                                        |
| YouTube      | YouTube (Cronet)                   |      22 | 145.094    |              14 |                4 |                  2 |            8 |                                                                                                         |                   1 | Advertising ID, GSF ID, Google Account Name                                                                            |
| YouTube      | Android System (Conscrypt)         |       5 |   9.22783  |               2 |                2 |                  2 |            3 |                                                                                                         |                   1 |                                                                                                                        |
| YouTube      | GMS (Conscrypt)                    |       5 |   5.53797  |               3 |                2 |                  0 |            2 |                                                                                                         |                   0 |                                                                                                                        |

,company,tls_lib,hosts,bytes_mb,captured_mitm,captured_frida,captured_aospsci,missed_all,missed_pii,missed_and_pinned,pii_types
108,ByteDance,TikTok (BoringSSL),43,223.141830,3,3,4,39,"Advertising ID, Android ID, Contact Email, Con...",8,"Advertising ID, Android ID, Contact Email, Con..."
107,ByteDance,CapCult (BoringSSL),28,227.629902,17,12,3,10,"Android ID, Gmail Profile Picture, Google Acco...",0,"Advertising ID, Android ID, Firebase Installat..."
104,ByteDance,Android WebView (Static BoringSSL),17,45.826613,8,6,8,5,,0,
105,ByteDance,GMS (Conscrypt),7,21.086008,4,5,5,2,,0,"Advertising ID, App Set ID"
106,ByteDance,Android System (Conscrypt),12,39.300997,7,7,9,2,App Set ID,0,"Advertising ID, App Set ID"
20,Google,Android WebView (Static BoringSSL),39,72.257664,23,15,14,13,"Advertising ID, Android ID, App Set ID, Google...",1,"Advertising ID, Android ID, App Set ID, Contac..."
21,Google,Android System (Conscrypt),25,19.552519,18,11,14,5,,0,"Gmail Profile Picture, Google Account Email, G..."
24,Google,CapCult (BoringSSL),6,0.000000,1,1,0,5,"Android ID, Gmail Profile Picture, Google Acco...",0,"Android ID, Gmail Profile Picture, Google Acco..."
23,Google,Adwords (Flutter),5,2.402562,3,3,2,2,Advertising ID,0,Advertising ID
25,Google,Snapchat (Static BoringSSL),3,0.916838,1,1,1,2,"Advertising ID, Android ID",0,"Advertising ID, Android ID"


Table breaks down the top data recipients by TLS library, reporting hosts observed by Advaita, hosts missed by other vantage points, traffic volume, per-vantage capture counts (Mitm/Frida/Aosp-Sci), and PII types found only in Advaita captures. Custom TLS libraries account for the largest coverage gaps. Snapchat's statically linked BoringSSL (14 hosts, 241.9 MB, 0/0/0) and Meta's Instagram Fizz stack (9 hosts, 216.4 MB, 0/0/0) are entirely invisible to all other vantage points, with Snapchat's missed traffic containing Google Advertising ID, SSAID, Contact Email, Contact Phone, Firebase Installation ID, and Google Account Name, while Meta's contains Google Advertising ID, SSAID, Photos Location, Router Location, and Video Location. TikTok's BoringSSL fork misses 39 of 43 hosts (223.1 MB), with similar PII types to Snapchat in the missed traffic. Among standard library users, Google AdMob misses 37 of 41 WebView BoringSSL hosts and 36 of 39 Conscrypt hosts, though traffic volumes are modest and no PII was detected in the missed traffic. Spotify's 6 missed Conscrypt hosts (267.1 MB) include SSAID transmissions, with a Frida capture count of 0 consistent with its known instrumentation detection. Uber's 2 missed WebView BoringSSL hosts carry a dense PII profile including Google Advertising ID, SSAID, App Set ID, Google Account Email, and Location despite the relatively small host count.


--------

## Summary

Consolidated findings for $\S$ 6.3.

Traffic gap: Of the 1,628 unique hosts contacted by 378 DRs, 729 (44.8%) were missed by all three existing techniques, accounting for 1133.67 MB (23.07%) of total traffic.

Custom TLS as primary mechanism: Of the 729 missed hosts, 138 (18.9%) use custom TLS exclusively, 412 (56.5%) use WebView exclusively, 142 (19.5%) use system Conscrypt exclusively, and 37 (5.1%) use a combination. System Conscrypt hosts are missed at a rate of 58.9%, compared to 81.9% for non-system hosts.

**Pinning ruled out: Only 37 of 729 missed hosts implement certificate pinning.**

Behavioral distortion: 139 DRs route traffic through multiple TLS libraries simultaneously. 167 missed hosts transmit PII, and 55 DRs transmit PII exclusively through non-system libraries.

### Scatter Plot: DR-Level Stream Counts (Figure 3)

In [ ]:
from IPython.display import Image
from analysis_utils.shared_setup import GAPS_DR, dr_stream_scatter, GAPS_MARKERS

endpoints_http = http_df[~http_df["host"].str.contains("gstatic")].copy()
coverage_mask = endpoints_http["company"].isin(GAPS_DR)

stream_pivot = (
    endpoints_http[coverage_mask]
    .groupby(["company", "vantage_type"])
    .size()
    .unstack(fill_value=0)
    .sort_values(list(endpoints_http["vantage_type"].unique()), ascending=False)
)

"""
B = Σ_c max(Aosp-Sci_c, Frida_c, Mitm_c)  =  3237 + 2021 + 1261 + 907 + 978 + 597 + 564 + 59 + 608 + 9 =  10,241
A = Σ_c Advaita_c                         =  29,440
miss = (A - B) / A = 19,199 / 29,440 = 65.2%
"""

stream_stat = stream_pivot.copy()
stream_stat["best_other"] = stream_stat[["Aosp-Sci", "Frida", "Mitm"]].max(axis=1)
miss_best = (stream_stat["Advaita"].sum() - stream_stat["best_other"].sum()) / stream_stat["Advaita"].sum() * 100
mark_print(f"**{miss_best:.1f}%** of all HTTP requests observed by Advaita for "
           f"{list(stream_stat.index)} were missed by other techniques.")

fig = dr_stream_scatter(stream_pivot, http_df["vantage_type"].unique(), font=dict(size=14))
# fig.write_image("../images/gaps.pdf", scale=1)
fig.show()
stream_pivot

**65.2%** of all HTTP requests observed by Advaita for ['Google AdMob', 'Google AdManager', 'Meta', 'ByteDance', 'Google', 'YouTube', 'Uber', 'Google Play Store', 'Spotify', 'Snapchat'] were missed by other techniques.

vantage_type,Advaita,Aosp-Sci,Frida,Mitm
company,,,,
Google AdMob,6809,137,2160,3237
Google AdManager,3326,966,1417,2021
Meta,5732,916,1261,1256
ByteDance,2700,424,780,907
Google,3569,978,448,963
YouTube,2140,14,354,597
Uber,1407,564,273,1
Google Play Store,322,59,3,2
Spotify,1836,34,1,608
